# 1. Importação e Configuração

Nesta etapa, são definidas as tabelas da camada Gold que serão publicadas no Google Planilhas (uma aba para cada tabela) e são validados os arquivos necessários: os CSVs da Gold e as credenciais de acesso.

Antes de rodar este notebook, é necessário configurar o acesso ao Google (conta de serviço) e salvar os dois arquivos na pasta `credenciais/`, que não deve ser enviada ao Git: a chave `google_service_account.json` e o arquivo `planilha_id.txt`, com o ID da planilha de destino.

In [1]:
# 1. Importação das bibliotecas e definição dos caminhos do projeto.
from pathlib import Path
import json
import pandas as pd
import gspread

RAIZ        = Path.cwd().parent
GOLD        = RAIZ / "data" / "Gold"
CREDENCIAIS = RAIZ / "credenciais"

# 2. Tabelas da camada Gold que serão publicadas (uma aba para cada uma).
TABELAS_GOLD = ["receita_mensal", "pagamentos_status_mensal", "assinaturas_mensal",
                "assinaturas_por_plano", "interacoes_por_tipo", "metadados", "kpis_resumo"]

# 3. Validação: Os arquivos da Gold e as credenciais precisam existir.
faltando = [t for t in TABELAS_GOLD if not (GOLD / f"{t}.csv").exists()]
assert not faltando, f"Faltam arquivos na Gold: {faltando}. Rode o notebook 07 antes do 08."
assert (CREDENCIAIS / "google_service_account.json").exists(), "Chave não encontrada em credenciais/google_service_account.json."
assert (CREDENCIAIS / "planilha_id.txt").exists(), "Arquivo credenciais/planilha_id.txt não encontrado."

# 2. Funções de leitura e preparação

Nesta etapa, são criadas duas funções: uma lê o CSV da Gold respeitando o formato em que ele foi gravado (separador `;` e decimal `,`), e a outra transforma a tabela no formato aceito pelo Planilhas, uma lista de linhas com o cabeçalho na primeira. As células vazias são enviadas como texto vazio.

In [2]:
# 1. Leitura: Os CSVs da Gold foram gravados com separador ";" e decimal ",".
def ler_gold(nome):
    return pd.read_csv(GOLD / f"{nome}.csv", sep=";", decimal=",", encoding="utf-8-sig")

# 2. Preparação: Cabeçalho + linhas, com os vazios (NaN) trocados por texto vazio.
## O Planilhas não aceita NaN, e isso evitaria erro na hora do envio.
def preparar_valores(df):
    corpo = df.astype(object).where(df.notna(), "")
    return [list(df.columns)] + corpo.values.tolist()

# 3. Conexão com o Google Planilhas

Nesta etapa, o notebook se autentica com a conta de serviço e abre a planilha de destino pelo ID.

In [3]:
# 1. Conexão: Autentica com a conta de serviço e abre a planilha pelo ID.
gc = gspread.service_account(filename=str(CREDENCIAIS / "google_service_account.json"))
planilha = gc.open_by_key((CREDENCIAIS / "planilha_id.txt").read_text().strip())
print("Planilha aberta:", planilha.title)

Planilha aberta: Gold Pandas 002


# 4. Envio das tabelas e conciliação

Nesta etapa, as tabelas da camada Gold são enviadas para o Google Planilhas, cada uma para a sua aba.

Primeiro, todas as tabelas são lidas e preparadas, sem tocar na planilha. Só depois, se tudo estiver certo, o envio começa. Para cada tabela, se a aba não existir ela é criada, o conteúdo antigo é apagado e o novo é gravado, então o resultado é sempre o mesmo, rodando 1 ou 10 vezes. Por fim, cada aba é lida de volta para conferir se o tamanho é o mesmo do CSV.

Atenção: o conteúdo das abas é substituído a cada execução. Não edite essas abas manualmente, pois as alterações serão perdidas.

In [4]:
# 1. Preparação: Todas as tabelas são lidas e preparadas ANTES de qualquer envio.
## Assim, um problema nos dados (por exemplo, um valor que o Planilhas não aceita) é descoberto antes de alterar a planilha.
pacotes = {}
for nome in TABELAS_GOLD:
    valores = preparar_valores(ler_gold(nome))
    json.dumps(valores, allow_nan=False)      # falha se existir NaN ou um tipo que o Planilhas não aceita
    pacotes[nome] = valores
print("Tabelas preparadas:", len(pacotes))

Tabelas preparadas: 7


In [5]:
# 2. Envio: Para cada tabela, abre a aba (ou cria, se não existir), limpa o conteúdo antigo e grava o novo.
## O value_input_option="RAW" grava os valores exatamente como enviados, sem o Planilhas tentar adivinhar o formato (datas continuam texto).
for nome, valores in pacotes.items():
    try:
        aba = planilha.worksheet(nome)
    except gspread.exceptions.WorksheetNotFound:
        aba = planilha.add_worksheet(title=nome, rows=len(valores) + 10, cols=len(valores[0]) + 2)

    aba.clear()
    aba.update(values=valores, range_name="A1", value_input_option="RAW")

    # 2.2 Conciliação: Lê a aba de volta e confere se o tamanho é o mesmo do CSV (cabeçalho + linhas).
    lido = aba.get_all_values()
    assert len(lido) == len(valores) and len(lido[0]) == len(valores[0]), f"{nome}: o tamanho na planilha difere do CSV."
    print(f"{nome:26s} {len(valores) - 1:4d} linhas -> OK")

print("Publicação concluída:", len(pacotes), "abas atualizadas.")

receita_mensal               33 linhas -> OK
pagamentos_status_mensal    132 linhas -> OK
assinaturas_mensal           33 linhas -> OK
assinaturas_por_plano         9 linhas -> OK
interacoes_por_tipo           5 linhas -> OK
metadados                     1 linhas -> OK
kpis_resumo                   1 linhas -> OK
Publicação concluída: 7 abas atualizadas.
